In [1]:
%load_ext autoreload
%autoreload 2

# Student Placement Prediction
## Objectif
L'objectif de ce projet est de développer un modèle de machine learning capable de prédire si un étudiant sera placé ou non après l'obtention de son diplôme.
## DataSet utiliser
Site Kaggle : https://www.kaggle.com/datasets/ashishpatel26/student-placement-prediction

**Nombre d'entrée**: 10,000
**nom colonne**:
- study_hours`, ` Nombre d'heures d'étude par semaine
- attendance`,  ` Taux de présence en pourcentage
- sleep_hours`,  ` Nombre d'heures de sommeil par nuit
- internet_usage`,  ` Temps passé sur Internet par jour en heures
- assignments_completed`, ` Nombre de devoirs complétés
- previous_score`,  ` Score précédent de l'étudiant
- exam_score ` Score de l'examen final
- placement_status ` Statut de placement

## Etape 1: Prétraitement des données
### Nettoyage des données et préparation
- Import les bibliothèque necessaire
- Charger le dataset

In [2]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join('..')))
from src.data_cleaning import (
    charger_donnees,
    traiter_valeurs_manquantes,
    encoder_cible,
    detecter_outliers,
    traiter_outliers,
    standardiser_donnees
)
from src.model_training import (
    entrainer_modeles,
    valider_modeles,
    selectionner_meilleur_modele
)
from src.visualization import (
    tracer_matrices_confusion,
    tracer_courbes_roc,
    tracer_importance_features,
    sauvegarder_rapports_classification,
)
import Config.config as config

df = charger_donnees(config.filepath)
print(df.head())
print("\n Statistiques descriptives:")
print(df.describe(include="all"))

Dataset chargé: 10000 lignes, 8 colonnes
   study_hours  attendance  sleep_hours  internet_usage  \
0            7          56            8               7   
1            4          69            5               3   
2           11          60            7               6   
3            8          99            9               8   
4            5          52            8               6   

   assignments_completed  previous_score  exam_score placement_status  
0                     10              62      100.00           Placed  
1                      8              56      100.00           Placed  
2                     10              45      100.00           Placed  
3                      4              55       90.17           Placed  
4                      8              40       78.82           Placed  

 Statistiques descriptives:
         study_hours   attendance   sleep_hours  internet_usage  \
count   10000.000000  10000.00000  10000.000000    10000.000000   
unique   

#### Supprimer les doublon et remplacer les valeurs manquantes
- Supprimer les doublons
- Résoudre les donner manquantes avec la médiane de la colonne

In [3]:
lignes_avant = len(df)
df = df.drop_duplicates()
print(f"Doublons supprimés: {lignes_avant - len(df)}")
df = traiter_valeurs_manquantes(df)

Doublons supprimés: 0
Valeurs manquantes traitées (médiane/mode)


#### Encoder la variable cible et identifier les variables numériques et catégorielles
- Encoder la variable cible
- Identifier les variables numériques et catégorielles

In [4]:
colonnes_features = [col for col in df.columns if col != config.colonne_cible]
x = df[colonnes_features].copy()
y, le = encoder_cible(df, config.colonne_cible)

colonnes_numeriques = x.select_dtypes(include=[np.number]).columns.tolist()
colonnes_categorielles = [
    col for col in x.columns if col not in colonnes_numeriques
]

print(f"\n Variables numériques: {colonnes_numeriques}")
print(f" Variables catégorielles: {colonnes_categorielles}")
print(f" Variable cible: {config.colonne_cible}")

Variable cible encodée: {'Not Placed': np.int64(0), 'Placed': np.int64(1)}

 Variables numériques: ['study_hours', 'attendance', 'sleep_hours', 'internet_usage', 'assignments_completed', 'previous_score', 'exam_score']
 Variables catégorielles: []
 Variable cible: placement_status


#### Trouver les outliers(valeur aberrant) et les traiter
- Pour trouver les outliers, on utilise la méthode de l'IQR (Interquartile Range) pour chaque variable numérique. Les points de données qui se trouvent en dehors de 1.5 fois l'IQR au-dessus du troisième quartile ou en dessous du premier quartile sont considérés comme des outliers.
- il sont remplacés par la médiane de la colonne pour éviter de fausser les statistiques et les modèles de machine learning.

In [5]:
detecter_outliers(x, colonnes_numeriques)
x = traiter_outliers(x, colonnes_numeriques)


 Détection des outliers (méthode IQR):
  - study_hours: 0 outliers détectés
  - attendance: 0 outliers détectés
  - sleep_hours: 0 outliers détectés
  - internet_usage: 0 outliers détectés
  - assignments_completed: 0 outliers détectés
  - previous_score: 0 outliers détectés
  - exam_score: 62 outliers détectés
Total: 62 outliers détectés
Outliers traités par clipping IQR


#### Encoder les variables catégorielles et standardiser les données
- Encoder les variables catégorielles avec la méthode one-hot encoding
- Standardiser les données numériques pour que toutes les caractéristiques soient sur la même échelle, ce qui est important pour de nombreux algorithmes de machine learning.

In [6]:
if colonnes_categorielles:
    x = pd.get_dummies(x, columns=colonnes_categorielles, drop_first=True)
    print("Variables catégorielles encodées (one-hot)")

x_scaled, scaler = standardiser_donnees(x)

print("\n Préparation des données terminée!")
print(f"Forme finale: X={x_scaled.shape}, y={y.shape}")

 Données standardisées (moyenne=0, std=1)

 Préparation des données terminée!
Forme finale: X=(10000, 7), y=(10000,)


### Etape 2: Entrainement du modèle
#### Premier model : regression linéaire
- fonctionnement : La régression linéaire est un algorithme de machine learning utilisé pour modéliser la relation entre une variable dépendante (cible) et une ou plusieurs variables indépendantes (caractéristiques). L'objectif est de trouver la meilleure ligne droite qui ajuste les données, permettant ainsi de faire des prédictions basées sur les caractéristiques d'entrée.
#### Deuxième model : Random Forest
- fonctionnement : Random Forest est un algorithme d'ensemble qui combine plusieurs arbres de décision pour améliorer la précision et réduire le risque de surapprentissage. Chaque arbre est construit à partir d'un échantillon aléatoire des données d'entraînement, et les prédictions finales sont obtenues en moyennant les prédictions de tous les arbres.
#### Troisième model : Support Vector Machine (SVM)
- fonctionnement : SVM est un algorithme de classification qui trouve l'hyperplan optimal qui sépare les classes dans l'espace des caractéristiques. Il maximise la marge entre les classes, ce qui le rend efficace pour les problèmes de classification binaire. SVM peut également être utilisé pour la régression (SVR) en trouvant une fonction qui s'écarte le moins possible des points de données tout en respectant une certaine marge d'erreur.

In [7]:
modeles, x_train, x_test, y_train, y_test = entrainer_modeles(x, y)


 Données divisées:
  - Train set: 8000 échantillons
  - Test set: 2000 échantillons

 Entraînement de 3 modèles...

Logistic Regression (Custom):
  Accuracy:  0.8885
  Precision: 0.8897
  Recall:    0.9892
  F1-Score:  0.9368

  Temps d'entraînement: 0.1021 secondes

Random Forest (Custom):
  Accuracy:  0.9995
  Precision: 1.0000
  Recall:    0.9994
  F1-Score:  0.9997

  Temps d'entraînement: 2.0803 secondes

Support Vector Machine (Custom):
  Accuracy:  0.9345
  Precision: 1.0000
  Recall:    0.9216
  F1-Score:  0.9592

  Temps d'entraînement: 19.1053 secondes



### Etape 3: Evaluation du modèle
#### Evaluation de la performance du modèle

In [8]:
resultats_cv = valider_modeles(modeles, x, y, cv=5)

➜ Logistic Regression (Custom):
  Scores par fold: [0.9144 0.9068 0.9387 0.9357 0.9033]
  Moyenne: 0.9198 (+/- 0.0147)

➜ Random Forest (Custom):
  Scores par fold: [0.9997 1.     0.9997 1.     1.    ]
  Moyenne: 0.9999 (+/- 0.0001)

➜ Support Vector Machine (Custom):
  Scores par fold: [0.9745 0.9623 0.9705 0.9717 0.988 ]
  Moyenne: 0.9734 (+/- 0.0084)



### Etape 4: Sélection du meilleur modèle
#### Sélectionner le meilleur modèle en fonction de la performance sur les données de test

In [9]:
meilleur_nom, meilleur_modele, meilleur_score = selectionner_meilleur_modele(modeles, x_test, y_test)

Meilleur modèle: Random Forest (Custom)
F1-Score: 0.9997

Classement:
1. Random Forest (Custom): 0.9997
2. Support Vector Machine (Custom): 0.9592
3. Logistic Regression (Custom): 0.9368


#### Etape 5: Graphique
- tout les graphique vont etre dans le dossier outputs
- Matrice de confusion pour chaque modèle
- Courbes ROC pour chaque modèle
- Importance des caractéristiques pour les modèles qui le permettent (ex: Random Forest)
- Rapports de classification pour chaque modèle

In [10]:
tracer_matrices_confusion(modeles, x_test, y_test, output_dir=config.output_dir)
tracer_courbes_roc(modeles, x_test, y_test, output_dir=config.output_dir)
tracer_importance_features(modeles, x_scaled.columns.tolist(), output_dir=config.output_dir)
sauvegarder_rapports_classification(modeles, x_test, y_test, output_dir=config.output_dir)

print(f"Meilleur modèle: {meilleur_nom} (F1={meilleur_score:.4f})")
print("Recommandation: utiliser ce modèle pour les prédictions finales.")
print(f"Visualisations et graphique dans le dossier: {config.output_dir}")

Meilleur modèle: Random Forest (Custom) (F1=0.9997)
Recommandation: utiliser ce modèle pour les prédictions finales.
Visualisations et graphique dans le dossier: ../outputs


## Conclusion
Dans le cadre de ce projet, nous avons développé et évalué trois modèles de machine learning différents (Régression Logistique, Random Forest, et Support Vector Machine (SVM)) afin de prédire le statut de placement des étudiants à partir d'un jeu de données comprenant 10 000 entrées.
L'analyse détaillée des rapports de classification et des performances globales permet de dresser un bilan précis du comportement de chaque algorithme sur les 2 000 échantillons du jeu de test.
#### classification des modèles
A. Régression Logistique
- La Régression Logistique est le moins performant des trois modèles, avec un F1-Score global de 0.87. Bien que le modèle excelle à identifier les étudiants placés (rappel de 99%), il montre une certaine difficulté à détecter les étudiants non placés, avec un rappel de seulement 38%. Cela suggère que la Régression Logistique a tendance à privilégier la classe majoritaire (placé) au détriment de la classe minoritaire (non placé) :
| Classe | Precision | Recall | F1-Score | Support |
|---------|----------:|--------:|---------:|---------:|
| Non Placé | 0.87 | 0.38 | 0.53 | 329 |
| Placé | 0.89 | 0.99 | 0.94 | 1671 |
| **Accuracy** |  |  | **0.89** | **2000** |
| **Macro Avg** | 0.88 | 0.68 | 0.73 | 2000 |
| **Weighted Avg** | 0.89 | 0.89 | 0.87 | 2000 |
B. Random Forest
- Le Random Forest s'illustre par un sans-faute absolu sur l'ensemble des métriques, parvenant à séparer parfaitement les deux classes de manière équivalente :
| Classe | Precision | Recall | F1-Score | Support |
|---------|----------:|--------:|---------:|---------:|
| Non Placé | 1.00 | 1.00 | 1.00 | 329 |
| Placé | 1.00 | 1.00 | 1.00 | 1671 |
| **Accuracy** |  |  | **1.00** | **2000** |
| **Macro Avg** | 1.00 | 1.00 | 1.00 | 2000 |
| **Weighted Avg** | 1.00 | 1.00 | 1.00 | 2000 |
C. Support Vector Machine (SVM)
- Bien que le SVM reste extrêmement performant (accuracy de 0.93), il est le deuxieme meilleur modèle de l'étude, montrant quelques hésitations sur l'identification des étudiants non placés (rappel de 72%) :
| Classe | Precision | Recall | F1-Score | Support |
|---------|----------:|--------:|---------:|---------:|
| Non Placé | 0.72 | 1.00 | 0.83 | 329 |
| Placé | 1.00 | 0.92 | 0.96 | 1671 |
| **Accuracy** |  |  | **0.93** | **2000** |
| **Macro Avg** | 0.86 | 0.96 | 0.90 | 2000 |
| **Weighted Avg** | 0.95 | 0.93 | 0.94 | 2000 |

#### Analyse comparative
| Indicateur clé | Régression Logistique |     Random Forest | Support Vector Machine (SVM) |
|---------------|----------------------:|------------------:|-----------------------------:|
| F1-Score Global (Test) |                0.9368 |            0.9997 |                       0.9592 |
| F1-Score Moyen (Cross-Validation) |     0.9198 (± 0.0147) | 0.9999 (± 0.0001) |            0.9734 (± 0.0084) |
| Temps d'entraînement |              0.0971 s |          2.0919 s |                    19.0061 s |
| Complexité computationnelle |           Très faible |           Modérée |                       Élevée |

1. Le Random Forest est mathématiquement le modèle le plus robuste de l'étude. Il obtient un score parfait de 1.0000. Cela démontre que le modèle ne subit aucune fluctuation selon la découpe des données. Il est un peu plus lent à entraîner que la Régression Logistique, mais il compense largement par sa capacité à capturer des relations complexes entre les caractéristiques.
2. La Régression Logistique, bien que très rapide à entraîner, atteint un score de 0.87, ce qui est nettement inférieur aux deux autres modèles. Son principal point faible réside dans sa difficulté à identifier les étudiants non placés, ce qui se traduit par un rappel de seulement 38% pour cette classe. Cependant, elle reste une option intéressante pour des scénarios où la rapidité et l'interprétabilité sont prioritaires.
3. Le Support Vector Machine (SVM) affiche une performance de 0.93, ce qui est très bon, mais il est le plus lent à entraîner. Il montre également une certaine difficulté à identifier les étudiants non placés, avec un rappel de 72%. Son utilisation peut être justifiée dans des contextes où la précision est cruciale et où les ressources de calcul ne sont pas une contrainte majeure.

#### Conclusion finale
Le modele Random Forest est clairement le meilleur choix pour ce projet, offrant une performance parfaite sur les données de test et une grande robustesse. La Régression Logistique, bien que rapide, n'est pas recommandée en raison de sa faible capacité à identifier les étudiants non placés. Le SVM, bien que performant, est moins efficace que le Random Forest et plus coûteux en termes de temps d'entraînement. Par conséquent, pour des prédictions finales, le Random Forest est la recommandation privilégiée.